In [1]:
# Target and Feature Engineering

## Leakage-Free Water-Stress Forecasting Dataset

This notebook constructs the monthly water-stress target and predictor features used in the one-month-ahead forecasting experiment.

The workflow consists of four main stages:

1. construction of monthly hydroclimatic water balance;
2. calculation of a standardized water-stress index using training-period climatology only;
3. generation of lagged, rolling, and seasonal predictor variables; and
4. temporal alignment of predictors at month *t* with the water-stress target at month *t+1*.

To prevent information leakage, climatological parameters used to standardize the water-stress index are estimated exclusively from the training period (2015–2021). Future validation and test observations therefore do not contribute to the definition of the target.

The resulting dataset forms the input to the feature-selection and Ridge-regression experiments conducted in the subsequent model-development notebook.

SyntaxError: invalid character '–' (U+2013) (1122659485.py, line 14)

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project data directory."
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

MONTHLY_FILE = (
    PROCESSED_DIR /
    "mnjoli_water_stress_monthly.csv"
)

ML_DATASET_FILE = (
    PROCESSED_DIR /
    "mnjoli_water_stress_ml_dataset.csv"
)

df_monthly = pd.read_csv(
    MONTHLY_FILE,
    parse_dates=["month_date"]
)

df_monthly = (
    df_monthly
    .sort_values("month_date")
    .reset_index(drop=True)
)

print("=" * 70)
print("MONTHLY HYDROCLIMATIC DATA")
print("=" * 70)

print("Shape:", df_monthly.shape)

print(
    "Period:",
    df_monthly["month_date"].min(),
    "to",
    df_monthly["month_date"].max()
)

print(
    "Missing values:",
    df_monthly.isna().sum().sum()
)

display(df_monthly.head())

MONTHLY HYDROCLIMATIC DATA
Shape: (132, 14)
Period: 2015-01-01 00:00:00 to 2025-12-01 00:00:00
Missing values: 0


,month_date,precipitation_mm,pet_mm,temperature_mean_c,temperature_min_c,temperature_max_c,dewpoint_c,soil_moisture_layer1,soil_moisture_layer2,runoff_mm,surface_runoff_mm,solar_radiation,wind_speed,surface_pressure_kpa
0,2015-01-01,108.399174,223.820343,24.244754,20.536907,29.074656,19.159322,0.254893,0.247987,0.000448,0.000436,2.080123e+07,1.792599,97.009791
1,2015-02-01,153.429877,205.196632,24.368461,19.930770,29.984754,18.570688,0.224155,0.216465,0.001506,0.001496,1.991791e+07,1.525280,96.886179
2,2015-03-01,66.667655,230.976277,24.528914,19.785820,30.321264,16.856486,0.150096,0.165199,0.000093,0.000083,1.899828e+07,1.499579,97.252747
3,2015-04-01,25.722274,158.938783,21.506630,17.276988,26.593867,15.441193,0.193402,0.175428,0.001043,0.001034,1.379342e+07,1.319609,97.352107
4,2015-05-01,6.401154,156.063646,20.217188,14.224896,26.989467,13.045341,0.155295,0.184315,0.000055,0.000046,1.443251e+07,1.215148,97.510695


In [3]:
# ============================================================
# WATER-BALANCE CONSTRUCTION
# ============================================================

df_monthly["water_balance_mm"] = (
    df_monthly["precipitation_mm"]
    - df_monthly["pet_mm"]
)

df_monthly["water_balance_3month"] = (
    df_monthly["water_balance_mm"]
    .rolling(
        window=3,
        min_periods=3
    )
    .sum()
)

print("=" * 70)
print("WATER-BALANCE CONSTRUCTION")
print("=" * 70)

print(
    "Missing monthly water balance:",
    df_monthly["water_balance_mm"].isna().sum()
)

print(
    "Missing 3-month water balance:",
    df_monthly["water_balance_3month"].isna().sum()
)

display(
    df_monthly[
        [
            "month_date",
            "precipitation_mm",
            "pet_mm",
            "water_balance_mm",
            "water_balance_3month"
        ]
    ].head(12)
)

WATER-BALANCE CONSTRUCTION
Missing monthly water balance: 0
Missing 3-month water balance: 2


,month_date,precipitation_mm,pet_mm,water_balance_mm,water_balance_3month
0,2015-01-01,108.399174,223.820343,-115.421169,NaN
1,2015-02-01,153.429877,205.196632,-51.766755,NaN
2,2015-03-01,66.667655,230.976277,-164.308621,-331.496546
3,2015-04-01,25.722274,158.938783,-133.216509,-349.291885
4,2015-05-01,6.401154,156.063646,-149.662492,-447.187622
5,2015-06-01,6.331618,144.229834,-137.898216,-420.777217
6,2015-07-01,10.037668,149.188829,-139.151162,-426.711869
7,2015-08-01,8.076374,180.066791,-171.990417,-449.039795
8,2015-09-01,33.993718,196.083262,-162.089543,-473.231122
9,2015-10-01,33.239156,275.266909,-242.027753,-576.107714


In [4]:
# ============================================================
# TRAINING-PERIOD CLIMATOLOGY
# ============================================================

TRAIN_START = pd.Timestamp("2015-01-01")
TRAIN_END = pd.Timestamp("2021-12-01")

df_monthly["calendar_month"] = (
    df_monthly["month_date"].dt.month
)

training_climatology_data = df_monthly.loc[
    (df_monthly["month_date"] >= TRAIN_START)
    & (df_monthly["month_date"] <= TRAIN_END)
].copy()

training_climatology = (
    training_climatology_data
    .groupby("calendar_month")[
        "water_balance_3month"
    ]
    .agg(["mean", "std"])
    .rename(
        columns={
            "mean": "wb3_train_mean",
            "std": "wb3_train_std"
        }
    )
    .reset_index()
)

print("=" * 70)
print("TRAINING-PERIOD CLIMATOLOGY")
print("=" * 70)

print(
    "Climatology period:",
    TRAIN_START,
    "to",
    TRAIN_END
)

display(
    training_climatology.round(4)
)

TRAINING-PERIOD CLIMATOLOGY
Climatology period: 2015-01-01 00:00:00 to 2021-12-01 00:00:00


,calendar_month,wb3_train_mean,wb3_train_std
0,1,-398.6579,158.9201
1,2,-287.6825,160.1828
2,3,-266.7649,155.2436
3,4,-277.7504,106.6305
4,5,-369.4874,43.9751
5,6,-372.8801,31.3989
6,7,-389.1788,43.4657
7,8,-438.6049,25.1806
8,9,-499.8706,24.0016
9,10,-543.2090,34.4270


In [7]:
# ============================================================
# LEAKAGE-FREE WATER-STRESS INDEX
# ============================================================

# ------------------------------------------------------------
# 1. REMOVE OLD/STale CLIMATOLOGY COLUMNS IF THEY EXIST
# ------------------------------------------------------------

columns_to_remove = [
    "wb3_train_mean",
    "wb3_train_std"
]

existing_columns_to_remove = [
    col for col in columns_to_remove
    if col in df_monthly.columns
]

if existing_columns_to_remove:
    df_monthly = df_monthly.drop(
        columns=existing_columns_to_remove
    )


# ------------------------------------------------------------
# 2. MERGE TRAINING-PERIOD CLIMATOLOGY
# ------------------------------------------------------------

df_monthly = df_monthly.merge(
    training_climatology,
    on="calendar_month",
    how="left",
    validate="many_to_one"
)

print("=" * 70)
print("CLIMATOLOGY MERGE CHECK")
print("=" * 70)

print(
    "wb3_train_mean exists:",
    "wb3_train_mean" in df_monthly.columns
)

print(
    "wb3_train_std exists:",
    "wb3_train_std" in df_monthly.columns
)

print(
    "Missing climatology means:",
    df_monthly["wb3_train_mean"].isna().sum()
)

print(
    "Missing climatology std:",
    df_monthly["wb3_train_std"].isna().sum()
)


# ------------------------------------------------------------
# 3. CONSTRUCT STANDARDIZED WATER-STRESS INDEX
# ------------------------------------------------------------

df_monthly["water_stress_index"] = (
    df_monthly["water_balance_3month"]
    - df_monthly["wb3_train_mean"]
) / df_monthly["wb3_train_std"]


# ------------------------------------------------------------
# 4. SUMMARY CHECKS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LEAKAGE-FREE WATER-STRESS INDEX")
print("=" * 70)

print(
    "Valid WSI observations:",
    df_monthly["water_stress_index"]
    .notna()
    .sum()
)

print(
    "Missing WSI observations:",
    df_monthly["water_stress_index"]
    .isna()
    .sum()
)

print("\nWSI summary:")

print(
    df_monthly["water_stress_index"]
    .describe()
    .round(4)
)


# ------------------------------------------------------------
# 5. FIRST VALID OBSERVATIONS
# ------------------------------------------------------------

print("\nFirst valid observations:")

preview = (
    df_monthly.loc[
        df_monthly["water_stress_index"].notna(),
        [
            "month_date",
            "water_balance_3month",
            "wb3_train_mean",
            "wb3_train_std",
            "water_stress_index"
        ]
    ]
    .head(12)
    .copy()
)

numeric_columns = (
    preview
    .select_dtypes(include=[np.number])
    .columns
)

preview[numeric_columns] = (
    preview[numeric_columns]
    .round(4)
)

display(preview)


# ------------------------------------------------------------
# 6. EXTREME VALUE CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WSI EXTREME VALUE CHECK")
print("=" * 70)

max_index = (
    df_monthly["water_stress_index"]
    .idxmax()
)

min_index = (
    df_monthly["water_stress_index"]
    .idxmin()
)

print("\nMaximum WSI observation:")

display(
    df_monthly.loc[
        [max_index],
        [
            "month_date",
            "water_balance_3month",
            "water_stress_index"
        ]
    ]
)

print("\nMinimum WSI observation:")

display(
    df_monthly.loc[
        [min_index],
        [
            "month_date",
            "water_balance_3month",
            "water_stress_index"
        ]
    ]
)

CLIMATOLOGY MERGE CHECK
wb3_train_mean exists: True
wb3_train_std exists: True
Missing climatology means: 0
Missing climatology std: 0

LEAKAGE-FREE WATER-STRESS INDEX
Valid WSI observations: 130
Missing WSI observations: 2

WSI summary:
count    130.0000
mean       0.3286
std        1.2718
min       -1.7669
25%       -0.4907
50%        0.1639
75%        1.0642
max        7.0614
Name: water_stress_index, dtype: float64

First valid observations:


,month_date,water_balance_3month,wb3_train_mean,wb3_train_std,water_stress_index
2,2015-03-01,-331.4965,-266.7649,155.2436,-0.4170
3,2015-04-01,-349.2919,-277.7504,106.6305,-0.6709
4,2015-05-01,-447.1876,-369.4874,43.9751,-1.7669
5,2015-06-01,-420.7772,-372.8801,31.3989,-1.5254
6,2015-07-01,-426.7119,-389.1788,43.4657,-0.8635
7,2015-08-01,-449.0398,-438.6049,25.1806,-0.4144
8,2015-09-01,-473.2311,-499.8706,24.0016,1.1099
9,2015-10-01,-576.1077,-543.2090,34.4270,-0.9556
10,2015-11-01,-654.2689,-509.0128,103.6945,-1.4008
11,2015-12-01,-681.5608,-450.7918,146.5579,-1.5746



WSI EXTREME VALUE CHECK

Maximum WSI observation:


,month_date,water_balance_3month,water_stress_index
112,2024-05-01,-58.962463,7.061382



Minimum WSI observation:


,month_date,water_balance_3month,water_stress_index
4,2015-05-01,-447.187622,-1.766915


In [8]:
# ============================================================
# TEMPORAL FEATURE ENGINEERING
# ============================================================

# Environmental variables for which 1-, 2-, and 3-month
# lagged predictors will be generated.
lag_variables = [
    "precipitation_mm",
    "pet_mm",
    "temperature_mean_c",
    "temperature_min_c",
    "temperature_max_c",
    "dewpoint_c",
    "soil_moisture_layer1",
    "soil_moisture_layer2",
    "runoff_mm",
    "surface_runoff_mm",
    "solar_radiation",
    "wind_speed",
    "surface_pressure_kpa"
]


# ------------------------------------------------------------
# 1. CREATE LAGGED PREDICTORS
# ------------------------------------------------------------

for variable in lag_variables:

    for lag in [1, 2, 3]:

        df_monthly[
            f"{variable}_lag{lag}"
        ] = (
            df_monthly[variable]
            .shift(lag)
        )


# ------------------------------------------------------------
# 2. CREATE ACCUMULATED PRECIPITATION VARIABLES
# ------------------------------------------------------------

df_monthly["precipitation_3month"] = (
    df_monthly["precipitation_mm"]
    .rolling(
        window=3,
        min_periods=3
    )
    .sum()
)

df_monthly["precipitation_6month"] = (
    df_monthly["precipitation_mm"]
    .rolling(
        window=6,
        min_periods=6
    )
    .sum()
)


# ------------------------------------------------------------
# 3. CREATE ACCUMULATED PET VARIABLES
# ------------------------------------------------------------

df_monthly["pet_3month"] = (
    df_monthly["pet_mm"]
    .rolling(
        window=3,
        min_periods=3
    )
    .sum()
)

df_monthly["pet_6month"] = (
    df_monthly["pet_mm"]
    .rolling(
        window=6,
        min_periods=6
    )
    .sum()
)


# ------------------------------------------------------------
# 4. CREATE CYCLICAL SEASONAL VARIABLES
# ------------------------------------------------------------

df_monthly["month_sin"] = np.sin(
    2
    * np.pi
    * df_monthly["calendar_month"]
    / 12
)

df_monthly["month_cos"] = np.cos(
    2
    * np.pi
    * df_monthly["calendar_month"]
    / 12
)


# ------------------------------------------------------------
# 5. FEATURE-ENGINEERING SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("TEMPORAL FEATURE ENGINEERING")
print("=" * 70)

print(
    "Dataset shape:",
    df_monthly.shape
)

print(
    "Original lag variables:",
    len(lag_variables)
)

print(
    "Lagged predictors created:",
    len(lag_variables) * 3
)

print(
    "Rolling predictors created:",
    4
)

print(
    "Seasonal predictors created:",
    2
)


# ------------------------------------------------------------
# 6. CHECK MISSING VALUES INTRODUCED
# ------------------------------------------------------------

missing_values = (
    df_monthly
    .isna()
    .sum()
)

missing_values = (
    missing_values[
        missing_values > 0
    ]
    .sort_values(
        ascending=False
    )
)

print("\nMissing values after feature engineering:")

if len(missing_values) == 0:

    print("NONE")

else:

    print(missing_values)


# ------------------------------------------------------------
# 7. DISPLAY SAMPLE ENGINEERED FEATURES
# ------------------------------------------------------------

engineered_preview_columns = [
    "month_date",

    "precipitation_mm",
    "precipitation_mm_lag1",
    "precipitation_mm_lag2",
    "precipitation_mm_lag3",

    "pet_mm",
    "pet_mm_lag1",
    "pet_mm_lag2",
    "pet_mm_lag3",

    "precipitation_3month",
    "precipitation_6month",

    "pet_3month",
    "pet_6month",

    "month_sin",
    "month_cos"
]

preview_features = (
    df_monthly[
        engineered_preview_columns
    ]
    .head(10)
    .copy()
)

# Round numeric columns only so datetime values are unaffected.
numeric_columns = (
    preview_features
    .select_dtypes(
        include=[np.number]
    )
    .columns
)

preview_features[numeric_columns] = (
    preview_features[numeric_columns]
    .round(4)
)

print("\nFirst engineered observations:")

display(
    preview_features
)

TEMPORAL FEATURE ENGINEERING
Dataset shape: (132, 69)
Original lag variables: 13
Lagged predictors created: 39
Rolling predictors created: 4
Seasonal predictors created: 2

Missing values after feature engineering:
pet_6month                   5
precipitation_6month         5
pet_mm_lag3                  3
temperature_mean_c_lag3      3
precipitation_mm_lag3        3
soil_moisture_layer1_lag3    3
dewpoint_c_lag3              3
runoff_mm_lag3               3
soil_moisture_layer2_lag3    3
temperature_max_c_lag3       3
temperature_min_c_lag3       3
surface_runoff_mm_lag3       3
solar_radiation_lag3         3
wind_speed_lag3              3
surface_pressure_kpa_lag3    3
precipitation_3month         2
surface_pressure_kpa_lag2    2
pet_mm_lag2                  2
temperature_mean_c_lag2      2
precipitation_mm_lag2        2
temperature_min_c_lag2       2
water_stress_index           2
water_balance_3month         2
pet_3month                   2
wind_speed_lag2              2
runoff_mm_

,month_date,precipitation_mm,precipitation_mm_lag1,precipitation_mm_lag2,precipitation_mm_lag3,pet_mm,pet_mm_lag1,pet_mm_lag2,pet_mm_lag3,precipitation_3month,precipitation_6month,pet_3month,pet_6month,month_sin,month_cos
0,2015-01-01,108.3992,NaN,NaN,NaN,223.8203,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500,0.866
1,2015-02-01,153.4299,108.3992,NaN,NaN,205.1966,223.8203,NaN,NaN,NaN,NaN,NaN,NaN,0.866,0.500
2,2015-03-01,66.6677,153.4299,108.3992,NaN,230.9763,205.1966,223.8203,NaN,328.4967,NaN,659.9933,NaN,1.000,0.000
3,2015-04-01,25.7223,66.6677,153.4299,108.3992,158.9388,230.9763,205.1966,223.8203,245.8198,NaN,595.1117,NaN,0.866,-0.500
4,2015-05-01,6.4012,25.7223,66.6677,153.4299,156.0636,158.9388,230.9763,205.1966,98.7911,NaN,545.9787,NaN,0.500,-0.866
5,2015-06-01,6.3316,6.4012,25.7223,66.6677,144.2298,156.0636,158.9388,230.9763,38.4550,366.9518,459.2323,1119.2255,0.000,-1.000
6,2015-07-01,10.0377,6.3316,6.4012,25.7223,149.1888,144.2298,156.0636,158.9388,22.7704,268.5902,449.4823,1044.5940,-0.500,-0.866
7,2015-08-01,8.0764,10.0377,6.3316,6.4012,180.0668,149.1888,144.2298,156.0636,24.4457,123.2367,473.4855,1019.4642,-0.866,-0.500
8,2015-09-01,33.9937,8.0764,10.0377,6.3316,196.0833,180.0668,149.1888,144.2298,52.1078,90.5628,525.3389,984.5711,-1.000,-0.000
9,2015-10-01,33.2392,33.9937,8.0764,10.0377,275.2669,196.0833,180.0668,149.1888,75.3092,98.0797,651.4170,1100.8993,-0.866,0.500


In [10]:
# ============================================================
# ONE-MONTH-AHEAD (t+1) FORECAST TARGET ALIGNMENT
# ============================================================

# ------------------------------------------------------------
# 1. ENSURE STRICT CHRONOLOGICAL ORDER
# ------------------------------------------------------------

df_monthly = (
    df_monthly
    .sort_values("month_date")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2. CREATE THE ONE-MONTH-AHEAD TARGET
# ------------------------------------------------------------
# Predictor values in row t are used to forecast the
# water-stress index in the following month (t+1).
#
# Example:
# predictors from 2024-01-01
#       ->
# WSI observed at 2024-02-01
# ------------------------------------------------------------

df_monthly["target_t_plus_1"] = (
    df_monthly["water_stress_index"]
    .shift(-1)
)


# ------------------------------------------------------------
# 3. CREATE THE CORRESPONDING FORECAST DATE
# ------------------------------------------------------------

df_monthly["forecast_date"] = (
    df_monthly["month_date"]
    .shift(-1)
)


# ------------------------------------------------------------
# 4. VERIFY ALIGNMENT
# ------------------------------------------------------------

print("=" * 70)
print("ONE-MONTH-AHEAD TARGET ALIGNMENT")
print("=" * 70)

print("Forecast horizon: 1 month ahead")

print(
    "Total monthly observations:",
    len(df_monthly)
)

print(
    "Valid t+1 targets:",
    df_monthly["target_t_plus_1"].notna().sum()
)

print(
    "Missing t+1 targets:",
    df_monthly["target_t_plus_1"].isna().sum()
)


# ------------------------------------------------------------
# 5. DISPLAY FIRST VALID ALIGNMENTS
# ------------------------------------------------------------

alignment_columns = [
    "month_date",
    "water_stress_index",
    "forecast_date",
    "target_t_plus_1"
]

first_alignment = (
    df_monthly[
        alignment_columns
    ]
    .dropna(
        subset=["target_t_plus_1"]
    )
    .head(12)
    .copy()
)

numeric_columns = (
    first_alignment
    .select_dtypes(include=[np.number])
    .columns
)

first_alignment[numeric_columns] = (
    first_alignment[numeric_columns]
    .round(4)
)

print("\nFirst valid forecast alignments:")

display(first_alignment)


# ------------------------------------------------------------
# 6. DISPLAY FINAL 12 ROWS
# ------------------------------------------------------------

last_alignment = (
    df_monthly[
        alignment_columns
    ]
    .tail(12)
    .copy()
)

numeric_columns = (
    last_alignment
    .select_dtypes(include=[np.number])
    .columns
)

last_alignment[numeric_columns] = (
    last_alignment[numeric_columns]
    .round(4)
)

print("\nFinal 12 forecast alignments:")

display(last_alignment)


# ------------------------------------------------------------
# 7. FINAL ALIGNMENT CHECKS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ALIGNMENT VERIFICATION")
print("=" * 70)

# Every non-final forecast date should be exactly the
# following monthly observation.
expected_forecast_dates = (
    df_monthly["month_date"]
    .shift(-1)
)

date_alignment_correct = (
    df_monthly["forecast_date"]
    .equals(expected_forecast_dates)
)

print(
    "Forecast-date alignment correct:",
    date_alignment_correct
)

print(
    "Final predictor month:",
    df_monthly["month_date"].iloc[-1]
)

print(
    "Final row forecast date:",
    df_monthly["forecast_date"].iloc[-1]
)

print(
    "Final row t+1 target:",
    df_monthly["target_t_plus_1"].iloc[-1]
)


# ------------------------------------------------------------
# 8. IMPORTANT SANITY CHECK
# ------------------------------------------------------------

print("\nExample interpretation:")

example = (
    df_monthly
    .dropna(subset=["target_t_plus_1"])
    .iloc[-1]
)

print(
    f"Predictors observed in {example['month_date'].strftime('%B %Y')}"
)

print(
    f"are used to forecast WSI for "
    f"{example['forecast_date'].strftime('%B %Y')}."
)

print(
    f"Observed target WSI = "
    f"{example['target_t_plus_1']:.4f}"
)

ONE-MONTH-AHEAD TARGET ALIGNMENT
Forecast horizon: 1 month ahead
Total monthly observations: 132
Valid t+1 targets: 130
Missing t+1 targets: 2

First valid forecast alignments:


,month_date,water_stress_index,forecast_date,target_t_plus_1
1,2015-02-01,NaN,2015-03-01,-0.4170
2,2015-03-01,-0.4170,2015-04-01,-0.6709
3,2015-04-01,-0.6709,2015-05-01,-1.7669
4,2015-05-01,-1.7669,2015-06-01,-1.5254
5,2015-06-01,-1.5254,2015-07-01,-0.8635
6,2015-07-01,-0.8635,2015-08-01,-0.4144
7,2015-08-01,-0.4144,2015-09-01,1.1099
8,2015-09-01,1.1099,2015-10-01,-0.9556
9,2015-10-01,-0.9556,2015-11-01,-1.4008
10,2015-11-01,-1.4008,2015-12-01,-1.5746



Final 12 forecast alignments:


,month_date,water_stress_index,forecast_date,target_t_plus_1
120,2025-01-01,0.1455,2025-02-01,0.6178
121,2025-02-01,0.6178,2025-03-01,0.6282
122,2025-03-01,0.6282,2025-04-01,1.1314
123,2025-04-01,1.1314,2025-05-01,1.1437
124,2025-05-01,1.1437,2025-06-01,1.0692
125,2025-06-01,1.0692,2025-07-01,-0.7045
126,2025-07-01,-0.7045,2025-08-01,-0.6790
127,2025-08-01,-0.6790,2025-09-01,1.0983
128,2025-09-01,1.0983,2025-10-01,0.8245
129,2025-10-01,0.8245,2025-11-01,1.9902



ALIGNMENT VERIFICATION
Forecast-date alignment correct: True
Final predictor month: 2025-12-01 00:00:00
Final row forecast date: NaT
Final row t+1 target: nan

Example interpretation:
Predictors observed in November 2025
are used to forecast WSI for December 2025.
Observed target WSI = 1.7403


In [12]:
# ============================================================
# CONSTRUCT FINAL LEAKAGE-FREE MODELING DATASET
# ============================================================

# ------------------------------------------------------------
# 1. DEFINE TARGET
# ------------------------------------------------------------

target_column = "target_t_plus_1"


# ------------------------------------------------------------
# 2. DEFINE NON-PREDICTOR / TARGET-DERIVED COLUMNS
# ------------------------------------------------------------
# These variables are deliberately excluded from model inputs.
#
# water_stress_index:
#     Current-month WSI. Excluded so that the forecasting model
#     is based on environmental predictors rather than simply
#     autoregressing the drought index.
#
# water_balance variables:
#     Direct components/intermediate quantities used to
#     construct the standardized water-stress index.
#
# wb3_train_*:
#     Training-period climatological statistics used directly
#     in target construction.
# ------------------------------------------------------------

non_predictor_columns = [
    # Dates
    "month_date",
    "forecast_date",

    # Target
    "target_t_plus_1",

    # Current target/index
    "water_stress_index",

    # Water-balance variables
    "water_balance_mm",
    "water_balance_3month",
    "water_balance_6month",

    # Training climatology
    "wb3_train_mean",
    "wb3_train_std",
    "wb3_train_mean_x",
    "wb3_train_std_x",
    "wb3_train_mean_y",
    "wb3_train_std_y"
]


# ------------------------------------------------------------
# 3. BUILD CANDIDATE PREDICTOR LIST
# ------------------------------------------------------------

predictor_columns = [
    column
    for column in df_monthly.columns
    if column not in non_predictor_columns
]


print("=" * 70)
print("FINAL PREDICTOR MATRIX CONSTRUCTION")
print("=" * 70)

print(
    "Candidate predictors:",
    len(predictor_columns)
)

print("\nCandidate predictor names:")

for i, feature in enumerate(
    predictor_columns,
    start=1
):
    print(
        f"{i:02d}. {feature}"
    )


# ------------------------------------------------------------
# 4. STRICT LEAKAGE CHECK
# ------------------------------------------------------------

forbidden_patterns = [
    "water_stress",
    "target_t_plus",
    "water_balance",
    "wb3_train"
]

leakage_found = [
    column
    for column in predictor_columns
    if any(
        pattern in column
        for pattern in forbidden_patterns
    )
]


print("\n" + "=" * 70)
print("STRICT LEAKAGE CHECK")
print("=" * 70)

if len(leakage_found) == 0:

    print(
        "PASS — no target-derived or water-balance "
        "variables are present in the predictor matrix."
    )

else:

    print(
        "FAIL — potential leakage variables detected:"
    )

    for variable in leakage_found:
        print(
            variable
        )


# ------------------------------------------------------------
# 5. CONSTRUCT MODELING DATAFRAME
# ------------------------------------------------------------

model_columns = (
    ["month_date", "forecast_date"]
    + predictor_columns
    + [target_column]
)

df_model = (
    df_monthly[
        model_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# 6. REMOVE OBSERVATIONS WITHOUT FUTURE TARGET
# ------------------------------------------------------------

rows_initial = len(df_model)

df_model = (
    df_model
    .dropna(
        subset=[target_column]
    )
    .reset_index(drop=True)
)

rows_after_target = len(df_model)


# ------------------------------------------------------------
# 7. REMOVE INCOMPLETE HISTORICAL FEATURE ROWS
# ------------------------------------------------------------
# Missing values at the beginning of the series are structural:
# lagged and rolling predictors require prior observations.
# These rows are therefore removed rather than imputed.
# ------------------------------------------------------------

df_model = (
    df_model
    .dropna(
        subset=predictor_columns
    )
    .reset_index(drop=True)
)

rows_final = len(df_model)


# ------------------------------------------------------------
# 8. TEMPORAL ALIGNMENT CHECK
# ------------------------------------------------------------

month_difference = (
    (
        df_model["forecast_date"].dt.year
        - df_model["month_date"].dt.year
    ) * 12
    +
    (
        df_model["forecast_date"].dt.month
        - df_model["month_date"].dt.month
    )
)

alignment_pass = (
    month_difference == 1
).all()


# ------------------------------------------------------------
# 9. FINAL DATASET VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MODELING DATASET")
print("=" * 70)

print(
    "Original monthly observations:",
    rows_initial
)

print(
    "Observations after valid-target filter:",
    rows_after_target
)

print(
    "Observations after predictor-history filter:",
    rows_final
)

print(
    "Final candidate predictors:",
    len(predictor_columns)
)

print(
    "Missing predictor values:",
    df_model[
        predictor_columns
    ]
    .isna()
    .sum()
    .sum()
)

print(
    "Missing target values:",
    df_model[
        target_column
    ]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# 10. FORECASTING PERIOD
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FORECASTING PERIOD")
print("=" * 70)

print(
    "First predictor month:",
    df_model["month_date"].min()
)

print(
    "First forecast month:",
    df_model["forecast_date"].min()
)

print(
    "Last predictor month:",
    df_model["month_date"].max()
)

print(
    "Last forecast month:",
    df_model["forecast_date"].max()
)

print(
    "Every observation represents t -> t+1:",
    alignment_pass
)


# ------------------------------------------------------------
# 11. FINAL SAFETY ASSERTIONS
# ------------------------------------------------------------
# If any of these conditions fail, execution stops immediately.
# ------------------------------------------------------------

assert len(leakage_found) == 0, (
    "Target-derived variables remain in predictor matrix."
)

assert alignment_pass, (
    "Temporal t -> t+1 alignment is incorrect."
)

assert (
    df_model[predictor_columns]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing predictor values remain."

assert (
    df_model[target_column]
    .isna()
    .sum()
    == 0
), "Missing target values remain."


# ------------------------------------------------------------
# 12. DISPLAY FIRST AND LAST OBSERVATIONS
# ------------------------------------------------------------

print("\nFirst 5 modeling observations:")

display(
    df_model[
        [
            "month_date",
            "forecast_date",
            target_column
        ]
    ]
    .head()
)

print("\nLast 5 modeling observations:")

display(
    df_model[
        [
            "month_date",
            "forecast_date",
            target_column
        ]
    ]
    .tail()
)

FINAL PREDICTOR MATRIX CONSTRUCTION
Candidate predictors: 59

Candidate predictor names:
01. precipitation_mm
02. pet_mm
03. temperature_mean_c
04. temperature_min_c
05. temperature_max_c
06. dewpoint_c
07. soil_moisture_layer1
08. soil_moisture_layer2
09. runoff_mm
10. surface_runoff_mm
11. solar_radiation
12. wind_speed
13. surface_pressure_kpa
14. calendar_month
15. precipitation_mm_lag1
16. precipitation_mm_lag2
17. precipitation_mm_lag3
18. pet_mm_lag1
19. pet_mm_lag2
20. pet_mm_lag3
21. temperature_mean_c_lag1
22. temperature_mean_c_lag2
23. temperature_mean_c_lag3
24. temperature_min_c_lag1
25. temperature_min_c_lag2
26. temperature_min_c_lag3
27. temperature_max_c_lag1
28. temperature_max_c_lag2
29. temperature_max_c_lag3
30. dewpoint_c_lag1
31. dewpoint_c_lag2
32. dewpoint_c_lag3
33. soil_moisture_layer1_lag1
34. soil_moisture_layer1_lag2
35. soil_moisture_layer1_lag3
36. soil_moisture_layer2_lag1
37. soil_moisture_layer2_lag2
38. soil_moisture_layer2_lag3
39. runoff_mm_lag1
4

,month_date,forecast_date,target_t_plus_1
0,2015-06-01,2015-07-01,-0.863509
1,2015-07-01,2015-08-01,-0.414403
2,2015-08-01,2015-09-01,1.109906
3,2015-09-01,2015-10-01,-0.955608
4,2015-10-01,2015-11-01,-1.400808



Last 5 modeling observations:


,month_date,forecast_date,target_t_plus_1
121,2025-07-01,2025-08-01,-0.679025
122,2025-08-01,2025-09-01,1.098294
123,2025-09-01,2025-10-01,0.824536
124,2025-10-01,2025-11-01,1.990155
125,2025-11-01,2025-12-01,1.740321


In [13]:
# ============================================================
# SAVE FINAL LEAKAGE-FREE MODELING DATASET
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1. DEFINE PROJECT PATHS
# ------------------------------------------------------------

current_dir = Path.cwd()

# Notebook is expected to run from:
# Eswatini_Water_Stress_ML/notebooks/
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

processed_dir = (
    project_root
    / "data"
    / "processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    processed_dir
    / "mnjoli_water_stress_modeling_dataset.csv"
)


# ------------------------------------------------------------
# 2. SAVE MODELING DATASET
# ------------------------------------------------------------

df_model.to_csv(
    output_path,
    index=False
)


# ------------------------------------------------------------
# 3. RELOAD FILE FOR INDEPENDENT VERIFICATION
# ------------------------------------------------------------

verification_df = pd.read_csv(
    output_path,
    parse_dates=[
        "month_date",
        "forecast_date"
    ]
)


# ------------------------------------------------------------
# 4. VERIFY SAVED DATASET
# ------------------------------------------------------------

print("=" * 70)
print("FINAL MODELING DATASET SAVED")
print("=" * 70)

print(
    "Output file:",
    output_path
)

print(
    "\nSaved shape:",
    verification_df.shape
)

print(
    "Observations:",
    len(verification_df)
)

print(
    "Candidate predictors:",
    len(predictor_columns)
)

print(
    "First predictor month:",
    verification_df["month_date"].min()
)

print(
    "First forecast month:",
    verification_df["forecast_date"].min()
)

print(
    "Last predictor month:",
    verification_df["month_date"].max()
)

print(
    "Last forecast month:",
    verification_df["forecast_date"].max()
)

print(
    "Missing values:",
    verification_df.isna().sum().sum()
)

print(
    "Duplicate predictor months:",
    verification_df["month_date"].duplicated().sum()
)

print(
    "Duplicate forecast months:",
    verification_df["forecast_date"].duplicated().sum()
)


# ------------------------------------------------------------
# 5. VERIFY TARGET COLUMN
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET VERIFICATION")
print("=" * 70)

print(
    verification_df["target_t_plus_1"]
    .describe()
    .round(4)
)


# ------------------------------------------------------------
# 6. SAFETY ASSERTIONS
# ------------------------------------------------------------

assert verification_df.shape[0] == 126, (
    "Unexpected number of observations."
)

assert verification_df.isna().sum().sum() == 0, (
    "Saved dataset contains missing values."
)

assert verification_df[
    "month_date"
].duplicated().sum() == 0, (
    "Duplicate predictor months detected."
)

assert verification_df[
    "forecast_date"
].duplicated().sum() == 0, (
    "Duplicate forecast months detected."
)

assert (
    verification_df["forecast_date"].min()
    == pd.Timestamp("2015-07-01")
), "Unexpected first forecast date."

assert (
    verification_df["forecast_date"].max()
    == pd.Timestamp("2025-12-01")
), "Unexpected final forecast date."


# ------------------------------------------------------------
# 7. CONFIRM SUCCESS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NOTEBOOK 03 COMPLETE")
print("=" * 70)

print(
    "PASS — leakage-free modeling dataset successfully saved."
)

print(
    "The dataset is ready for chronological model development."
)

FINAL MODELING DATASET SAVED
Output file: c:\Users\Para\Desktop\Eswatini_Water_Stress_ML\data\processed\mnjoli_water_stress_modeling_dataset.csv

Saved shape: (126, 62)
Observations: 126
Candidate predictors: 59
First predictor month: 2015-06-01 00:00:00
First forecast month: 2015-07-01 00:00:00
Last predictor month: 2025-11-01 00:00:00
Last forecast month: 2025-12-01 00:00:00
Missing values: 0
Duplicate predictor months: 0
Duplicate forecast months: 0

TARGET VERIFICATION
count    126.0000
mean       0.3738
std        1.2617
min       -1.5952
25%       -0.4585
50%        0.1826
75%        1.0910
max        7.0614
Name: target_t_plus_1, dtype: float64

NOTEBOOK 03 COMPLETE
PASS — leakage-free modeling dataset successfully saved.
The dataset is ready for chronological model development.
